# Mlflow Sandbox

## Imports

In [1]:
import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
import mlflow.sklearn

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import numpy as np

/Users/bhekimaenetja/.local/share/virtualenvs/small-projects-ai-NRjJWIjk/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Dummy Experiment

In [2]:
# Enable autologging to capture parameters, metrics, and models
mlflow.sklearn.autolog()

In [3]:
# Load sample data (replace with your actual data)
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([2, 4, 5, 4, 5])
print(f"X: {X}")
print(f"y: {y}")

X: [[1]
 [2]
 [3]
 [4]
 [5]]
y: [2 4 5 4 5]


In [4]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train: {X_train}")
print(f"X_test: {X_test}")
print(f"y_train: {y_train}")
print(f"y_test: {X_test}")

X_train: [[5]
 [3]
 [1]
 [4]]
X_test: [[2]]
y_train: [5 5 2 4]
y_test: [[2]]


In [5]:
# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

2025/08/11 16:11:46 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4935cbe6bb4d4994b2c90f5fedc9ffcf', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


LinearRegression()

In [6]:
# Make predictions on the test set
preds = model.predict(X_test)
preds

array([3.14285714])

In [7]:
# Evaluate the model
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"RMSE: {rmse}")

RMSE: 0.8571428571428572


In [8]:
# Log the model using MLflow
with mlflow.start_run() as run:
    # mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, "model")
    print(f"Run ID: {run.info.run_id}")

2025/08/11 16:11:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/08/11 16:11:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run ID: 7bec57b963bb4157a6cc56baaff70f04


In [9]:
run_id = "7bec57b963bb4157a6cc56baaff70f04"

# Register the model
model_uri = f"runs:/{run_id}/model"
model_name = "2nd_MLFlow_LR_Model"
registered_model_name = mlflow.register_model(model_uri, model_name)

print(f"Registered model name: {registered_model_name}")

Successfully registered model '2nd_MLFlow_LR_Model'.
2025/08/11 16:12:11 WARNING mlflow.tracking._model_registry.fluent: Run with id 7bec57b963bb4157a6cc56baaff70f04 has no artifacts at artifact path 'model', registering model based on models:/m-b616427a5e5e4470a2ac6af75fafb976 instead


Registered model name: <ModelVersion: aliases=[], creation_timestamp=1754925131539, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1754925131539, metrics=[], model_id='m-b616427a5e5e4470a2ac6af75fafb976', name='2nd_MLFlow_LR_Model', params={}, run_id='7bec57b963bb4157a6cc56baaff70f04', run_link=None, source='models:/m-b616427a5e5e4470a2ac6af75fafb976', status='READY', status_message=None, tags={}, user_id=None, version=1>


Created version '1' of model '2nd_MLFlow_LR_Model'.


In [10]:
# Initialize the MLflow client
client = MlflowClient()

# Transition the model to the "Production" stage
model_name = "2nd_MLFlow_LR_Model"
version = 1  # Replace with the actual version number
client.transition_model_version_stage(
    name=model_name,
    version=version,
    stage="Production"
)

# Load the model from the "Production" stage
model_uri = f"models:/{model_name}/Production"
model = mlflow.pyfunc.load_model(model_uri)

# Make predictions using the loaded model
X_new = np.array([[6]])  # Example input
predictions = model.predict(X_new)
print(f"Predictions: {predictions}")

Predictions: [5.88571429]


/var/folders/w5/rdwqly_s1bb_2klwzk3vbg240000gn/T/ipykernel_13051/2221702984.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
